# LeetCode 843 — Guess the Word bằng Python

- Link: https://leetcode.com/problems/guess-the-word/
- Độ khó: **Hard**
- Chủ đề: Bài tương tác (interactive), Lọc ứng viên nhất quán, Minimax, Heuristic, Mô phỏng lớp `Master`

> **Ghi chú:** đây là bài **tương tác** — trên LeetCode, đối tượng `master` do hệ thống cung cấp. Trong notebook ta **tự dựng lớp `Master` mô phỏng** để chạy và kiểm thử. Giới hạn số lần đoán (`allowedGuesses`) đã thay đổi qua các phiên bản đề (trước đây cố định 10 lần với 100 từ); notebook viết theo phiên bản hiện tại theo hiểu biết của tác giả (`allowedGuesses` được cho trong test, thường là 10) — nên đối chiếu lại trên trang bài.

Skeleton trên LeetCode (Python3):

```python
# """
# This is Master's API interface.
# You should not implement it, or speculate about its implementation
# """
# class Master:
#     def guess(self, word: str) -> int:

class Solution:
    def findSecretWord(self, words: List[str], master: 'Master') -> None:
```

## 1. Giới thiệu & mục tiêu học

Có một danh sách `words` gồm các từ **khác nhau**, mỗi từ đúng **6 chữ cái**. Một từ trong đó là **bí mật**. Bạn gọi `master.guess(w)`:

- Nếu `w` không có trong `words` → trả `-1`.
- Ngược lại → trả **số vị trí khớp** (cùng chữ ở cùng vị trí) giữa `w` và từ bí mật.

Bạn phải đoán trúng (nhận được `6`) trong tối đa `allowedGuesses` lần. Không có "đáp án trả về" — chỉ cần gọi `guess` đúng từ bí mật trước khi hết lượt.

Ý tưởng then chốt: sau mỗi lần đoán `w` nhận kết quả `k`, từ bí mật phải là một từ có `match(w, ·) == k` → **loại bỏ** mọi ứng viên không nhất quán. Câu hỏi khó là **chọn từ nào để đoán** sao cho loại được nhiều nhất trong trường hợp xấu nhất — đó là **minimax**.

Sau notebook này bạn sẽ:

1. Tự xây lớp `Master` mô phỏng để chạy bài tương tác.
2. Hiểu chiến lược **lọc ứng viên nhất quán** và vì sao nó không bao giờ loại nhầm từ bí mật.
3. So sánh chọn từ đoán ngẫu nhiên, theo tần suất chữ cái, và theo **minimax**.
4. Đánh giá chiến lược bằng **số lần đoán trong trường hợp xấu nhất** trên nhiều từ bí mật.
5. Biết cách viết kiểm thử cho thuật toán có yếu tố ngẫu nhiên (cố định seed).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `words` (các từ khác nhau, độ dài 6, chữ thường), một từ bí mật `secret ∈ words`, và số lượt `allowedGuesses`. Gọi `master.guess(word)` trả `-1` nếu `word ∉ words`, ngược lại trả số chỉ số `i` mà `word[i] == secret[i]`. Bạn thắng nếu gọi `master.guess(secret)` trong không quá `allowedGuesses` lần. (Kết quả trên LeetCode: "You guessed the secret word correctly." hoặc "Either you took too many guesses, or you did not find the secret word.")

**Ví dụ 1**

```text
Input: secret = "acckzz", words = ["acckzz","ccbazz","eiowzz","abcczz"], allowedGuesses = 10
Output: You guessed the secret word correctly.
Giải thích:
master.guess("aaaaaa") trả -1 vì "aaaaaa" không có trong words.
master.guess("acckzz") trả 6 → đoán trúng.
master.guess("ccbazz") trả 3.
master.guess("eiowzz") trả 2.
master.guess("abcczz") trả 4.
```

**Ví dụ 2**

```text
Input: secret = "hamada", words = ["hamada","khaled"], allowedGuesses = 10
Output: You guessed the secret word correctly.
```

**Ràng buộc** (theo phiên bản hiện tại)

- `1 <= words.length <= 100`
- `words[i].length == 6`, chữ thường, các từ khác nhau.
- `secret` có trong `words`.
- `10 <= allowedGuesses <= 30`

In [1]:
from typing import List

class Master:
    """Lớp mô phỏng API của LeetCode."""
    def __init__(self, words, secret, allowed_guesses=10):
        self.words = set(words)
        self.secret = secret
        self.allowed = allowed_guesses
        self.calls = 0
        self.solved = False
        self.log = []

    def guess(self, word: str) -> int:
        self.calls += 1
        if word not in self.words:
            res = -1
        else:
            res = sum(a == b for a, b in zip(word, self.secret))
        if res == 6 and self.calls <= self.allowed:
            self.solved = True
        self.log.append((word, res))
        return res

    def verdict(self):
        if self.solved:
            return "You guessed the secret word correctly."
        return "Either you took too many guesses, or you did not find the secret word."

m = Master(["acckzz", "ccbazz", "eiowzz", "abcczz"], "acckzz")
print([m.guess(w) for w in ["aaaaaa", "acckzz", "ccbazz", "eiowzz", "abcczz"]], m.verdict())

EXAMPLES = [
    (["acckzz", "ccbazz", "eiowzz", "abcczz"], "acckzz", 10),
    (["hamada", "khaled"], "hamada", 10),
    (["abcdef"], "abcdef", 10),
]

[-1, 6, 3, 2, 4] You guessed the secret word correctly.


## 3. Kiến thức nền

### 3.1. Hàm `match` — số vị trí khớp

In [2]:
def match(a, b):
    return sum(x == y for x, y in zip(a, b))

print(match("acckzz", "ccbazz"), match("acckzz", "abcczz"), match("acckzz", "acckzz"))

3 4 6


### 3.2. Lọc danh sách bằng list comprehension

Sau khi đoán `g` được `k`, giữ lại `[w for w in cand if match(w, g) == k]`.

In [3]:
cand = ["acckzz", "ccbazz", "eiowzz", "abcczz"]
g, k = "ccbazz", 3
print([w for w in cand if match(w, g) == k])

['acckzz']


### 3.3. Phân nhóm theo kết quả với `Counter`

Với một từ đoán `g`, các ứng viên chia thành 7 nhóm theo `match(g, w) ∈ {0..6}`. Kích thước nhóm lớn nhất là số ứng viên còn lại trong **trường hợp xấu nhất** nếu đoán `g`.

In [4]:
from collections import Counter

def partition_sizes(g, cand):
    return Counter(match(g, w) for w in cand)

print(partition_sizes("ccbazz", cand))

Counter({2: 2, 3: 1, 6: 1})


### 3.4. `random` có seed — tái lập được

Thuật toán có chọn ngẫu nhiên phải cố định seed khi kiểm thử để kết quả lặp lại.

In [5]:
import random
r1, r2 = random.Random(7), random.Random(7)
print([r1.randrange(100) for _ in range(5)], [r2.randrange(100) for _ in range(5)])

[41, 19, 50, 83, 6] [41, 19, 50, 83, 6]


### 3.5. `min` với `key` — chọn phần tử tốt nhất theo tiêu chí

In [6]:
words_demo = ["apple", "fig", "banana"]
print(min(words_demo, key=len), min(words_demo, key=lambda w: (-w.count('a'), w)))

fig banana


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Lọc nhất quán — tính tay ví dụ 1

Ứng viên ban đầu: `{acckzz, ccbazz, eiowzz, abcczz}`. Đoán `"ccbazz"`, nhận `3`.

| ứng viên | match với ccbazz | giữ? |
|---|---|---|
| acckzz | 3 (c ở vị trí 1, zz) | ✔ |
| ccbazz | 6 | ✗ |
| eiowzz | 2 | ✗ |
| abcczz | 2 | ✗ |

→ Còn 1 ứng viên `acckzz` → đoán, trúng. Tổng 2 lần.

### 4.2. Chọn từ đoán tốt

Nếu đoán `"eiowzz"`: phân nhóm `{2: [acckzz, ccbazz, abcczz], 6: [eiowzz]}` → xấu nhất còn 3. Đoán `"ccbazz"`: `{3: [acckzz], 6: [ccbazz], 2: [eiowzz, abcczz]}` → xấu nhất còn 2. Vậy `ccbazz` tốt hơn theo tiêu chí minimax.

### 4.3. Vì sao bài khó

Với 100 từ ngẫu nhiên, xác suất hai từ có `0` vị trí khớp là `(25/26)⁶ ≈ 79%`. Đoán ngẫu nhiên thường nhận `0` → chỉ loại khoảng 21% ứng viên! Chiến lược tốt phải chọn từ khiến nhóm `0` (và các nhóm khác) **nhỏ**.

In [7]:
for g in cand:
    print(g, dict(partition_sizes(g, cand)), "| xấu nhất:", max(c for k, c in partition_sizes(g, cand).items() if k != 6) if len(cand) > 1 else 0)

acckzz {6: 1, 3: 1, 2: 1, 4: 1} | xấu nhất: 1
ccbazz {3: 1, 6: 1, 2: 2} | xấu nhất: 2
eiowzz {2: 3, 6: 1} | xấu nhất: 3
abcczz {4: 1, 2: 2, 6: 1} | xấu nhất: 2


## 5. Cách 1 — Brute Force: đoán lần lượt từng từ

**Ý tưởng.** Gọi `guess` với từng từ theo thứ tự cho tới khi nhận `6`. Không dùng thông tin trả về.

In [8]:
def find_brute(words, master):
    for w in words:
        if master.guess(w) == 6:
            return

def run(strategy, words, secret, allowed=10):
    m = Master(words, secret, allowed)
    strategy(words, m)
    return m.solved, m.calls

for words, secret, allowed in EXAMPLES:
    print(run(find_brute, words, secret, allowed))

(True, 1)
(True, 1)
(True, 1)


**Độ phức tạp.** `O(n)` lần đoán trong trường hợp xấu nhất (`n` tới 100) — vượt xa 10 lượt. Chỉ thắng khi từ bí mật nằm trong 10 từ đầu.

**Nút thắt.** Bỏ phí thông tin `match` trả về. Mỗi kết quả cho biết từ bí mật nằm trong một "nhóm" cụ thể — ta phải loại bỏ những từ không thể là bí mật.

In [9]:
rnd = random.Random(843)
def random_words(n, rnd, alpha="abcdefghijklmnopqrstuvwxyz"):
    s = set()
    while len(s) < n:
        s.add(''.join(rnd.choice(alpha) for _ in range(6)))
    return sorted(s)

W100 = random_words(100, rnd)
worst = max(run(find_brute, W100, s)[1] for s in W100)
print("Brute force trên 100 từ: số lần đoán xấu nhất =", worst)

Brute force trên 100 từ: số lần đoán xấu nhất = 100


## 6. Bước đột phá

**Insight 1 — Lọc nhất quán không bao giờ loại nhầm.** Nếu đoán `g` được `k`, thì `match(g, secret) = k`. Do đó `secret` luôn nằm trong `{w : match(g, w) = k}`. Lọc theo điều kiện này an toàn.

**Insight 2 — Chỉ đoán từ trong danh sách.** Đoán từ ngoài danh sách trả `-1`, không có thông tin. Đoán **ứng viên** còn lại có thêm cơ hội trúng ngay.

**Insight 3 — Minimax.** Chọn `g` sao cho **nhóm lớn nhất** sau khi đoán là **nhỏ nhất**: `g* = argmin_g max_k |{w ∈ cand : match(g, w) = k, w ≠ g}|`. Đảm bảo giảm nhanh trong trường hợp xấu nhất. Với 100 từ ngẫu nhiên trên 26 chữ cái, lượt đầu thường chỉ chia được thành hai nhóm cỡ 2/3 và 1/3 (đa số cặp từ không khớp vị trí nào), nhưng các lượt sau co rất nhanh: thực nghiệm ở mục 11 cho trung bình ~5–6 lượt, xấu nhất ~9 — vừa đủ trong 10 lượt.

**Insight 4 — Heuristic rẻ hơn.** Thay vì minimax `O(n²)` mỗi lượt, có thể chấm điểm từ bằng tần suất chữ cái ở từng vị trí: từ có chữ "phổ biến" ở nhiều vị trí thì ít khả năng nhận `0` → nhóm `0` nhỏ.

**Quy tắc nếu…thì:**
- *Nếu* nhận `6` *thì* dừng.
- *Nếu* chỉ còn 1 ứng viên *thì* đoán nó.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đoán ngẫu nhiên một ứng viên + lọc

In [10]:
def find_random(words, master, seed=0):
    rng = random.Random(seed)
    cand = list(words)
    while cand:
        g = rng.choice(cand)
        k = master.guess(g)
        if k == 6:
            return
        cand = [w for w in cand if w != g and match(w, g) == k]

for words, secret, allowed in EXAMPLES:
    assert run(find_random, words, secret, allowed)[0]
stats = [run(find_random, W100, s)[1] for s in W100]
print("Ngẫu nhiên + lọc: xấu nhất", max(stats), "| trung bình %.2f" % (sum(stats) / len(stats)),
      "| tỉ lệ thắng (<=10):", sum(c <= 10 for c in stats), "/ 100")

Ngẫu nhiên + lọc: xấu nhất 12 | trung bình 6.38 | tỉ lệ thắng (<=10): 97 / 100


Code nộp LeetCode cho cách 2 (thường đủ với dữ liệu ngẫu nhiên, nhưng có thể trượt với test được thiết kế xấu):

In [11]:
class SolutionRandom:
    def findSecretWord(self, words: List[str], master: 'Master') -> None:
        def match(a, b):
            return sum(x == y for x, y in zip(a, b))
        cand = list(words)
        rng = random.Random(0)
        while cand:
            g = rng.choice(cand)
            k = master.guess(g)
            if k == 6:
                return
            cand = [w for w in cand if w != g and match(w, g) == k]

for words, secret, allowed in EXAMPLES:
    m = Master(words, secret, allowed); SolutionRandom().findSecretWord(words, m); assert m.solved
print("SolutionRandom: OK")

SolutionRandom: OK


**Độ phức tạp cách 2:** mỗi lượt `O(n·L)` để lọc (`L = 6`), tổng `O(G·n·L)` với `G` = số lần đoán.

### 7.2. Cách 3 — Minimax (chọn từ cực tiểu hoá nhóm lớn nhất)

Với mỗi ứng viên `g`, tính phân nhóm của các ứng viên khác theo `match(g, ·)`, lấy nhóm lớn nhất. Chọn `g` có giá trị nhỏ nhất (hoà thì lấy theo thứ tự để tất định).

In [12]:
def best_minimax(cand):
    best, best_score = None, None
    for g in cand:
        cnt = [0] * 7
        for w in cand:
            if w != g:
                cnt[match(g, w)] += 1
        score = max(cnt)
        if best_score is None or score < best_score:
            best, best_score = g, score
    return best

def find_minimax(words, master):
    cand = list(words)
    while cand:
        g = best_minimax(cand)
        k = master.guess(g)
        if k == 6:
            return
        cand = [w for w in cand if w != g and match(w, g) == k]

for words, secret, allowed in EXAMPLES:
    assert run(find_minimax, words, secret, allowed)[0]
stats = [run(find_minimax, W100, s)[1] for s in W100]
print("Minimax: xấu nhất", max(stats), "| trung bình %.2f" % (sum(stats) / len(stats)))

Minimax: xấu nhất 9 | trung bình 5.62


Code nộp LeetCode cho cách 3:

In [13]:
class SolutionMinimax:
    def findSecretWord(self, words: List[str], master: 'Master') -> None:
        def match(a, b):
            return sum(x == y for x, y in zip(a, b))
        cand = list(words)
        while cand:
            best, best_score = cand[0], 10 ** 9
            for g in cand:
                cnt = [0] * 7
                for w in cand:
                    if w != g:
                        cnt[match(g, w)] += 1
                if max(cnt) < best_score:
                    best, best_score = g, max(cnt)
            k = master.guess(best)
            if k == 6:
                return
            cand = [w for w in cand if w != best and match(w, best) == k]

for words, secret, allowed in EXAMPLES:
    m = Master(words, secret, allowed); SolutionMinimax().findSecretWord(words, m); assert m.solved
print("SolutionMinimax: OK")

SolutionMinimax: OK


**Độ phức tạp cách 3:** mỗi lượt `O(c²·L)` với `c` = số ứng viên; lượt đầu `100²·6 = 6·10⁴` — rất nhẹ. Tổng `O(G·n²·L)`.

### 7.3. Cách 4 — Heuristic tần suất chữ cái theo vị trí

Đếm `freq[i][ch]` = số ứng viên có chữ `ch` ở vị trí `i`. Điểm của từ `g` = `Σ_i freq[i][g[i]]` = tổng số "vị trí khớp" của `g` với mọi ứng viên. Chọn từ điểm cao nhất — trung bình khớp nhiều → ít khả năng nhận `0`.

In [14]:
def best_freq(cand):
    freq = [Counter(w[i] for w in cand) for i in range(6)]
    return max(cand, key=lambda g: (sum(freq[i][g[i]] for i in range(6)), g))

def find_freq(words, master):
    cand = list(words)
    while cand:
        g = best_freq(cand)
        k = master.guess(g)
        if k == 6:
            return
        cand = [w for w in cand if w != g and match(w, g) == k]

for words, secret, allowed in EXAMPLES:
    assert run(find_freq, words, secret, allowed)[0]
stats = [run(find_freq, W100, s)[1] for s in W100]
print("Tần suất chữ cái: xấu nhất", max(stats), "| trung bình %.2f" % (sum(stats) / len(stats)))

Tần suất chữ cái: xấu nhất 10 | trung bình 5.53


Code nộp LeetCode cho cách 4:

In [15]:
class SolutionFreq:
    def findSecretWord(self, words: List[str], master: 'Master') -> None:
        def match(a, b):
            return sum(x == y for x, y in zip(a, b))
        cand = list(words)
        while cand:
            freq = [Counter(w[i] for w in cand) for i in range(6)]
            g = max(cand, key=lambda x: sum(freq[i][x[i]] for i in range(6)))
            k = master.guess(g)
            if k == 6:
                return
            cand = [w for w in cand if w != g and match(w, g) == k]

for words, secret, allowed in EXAMPLES:
    m = Master(words, secret, allowed); SolutionFreq().findSecretWord(words, m); assert m.solved
print("SolutionFreq: OK")

SolutionFreq: OK


**Độ phức tạp cách 4:** mỗi lượt `O(c·L)` — tuyến tính, nhanh hơn minimax.

### 7.4. Cách 5 — Minimax trên **mọi từ** (không chỉ ứng viên)

Đôi khi từ đã bị loại lại phân nhóm các ứng viên tốt hơn bất kỳ ứng viên nào. Đổi lại, đoán từ đó **không thể trúng**. Ta chọn từ trong toàn bộ `words` với tiêu chí `(nhóm lớn nhất, không phải ứng viên)` — ưu tiên ứng viên khi hoà.

In [16]:
def best_minimax_all(words, cand):
    cset = set(cand)
    best, best_key = None, None
    for g in words:
        cnt = [0] * 7
        for w in cand:
            if w != g:
                cnt[match(g, w)] += 1
        key = (max(cnt), g not in cset)
        if best_key is None or key < best_key:
            best, best_key = g, key
    return best

def find_minimax_all(words, master):
    cand = list(words)
    guessed = set()
    while cand:
        g = cand[0] if len(cand) == 1 else best_minimax_all(words, cand)
        if g in guessed:                      # an toàn: không đoán lặp
            g = cand[0]
        guessed.add(g)
        k = master.guess(g)
        if k == 6:
            return
        cand = [w for w in cand if w != g and match(w, g) == k]

for words, secret, allowed in EXAMPLES:
    assert run(find_minimax_all, words, secret, allowed)[0]
stats = [run(find_minimax_all, W100, s)[1] for s in W100]
print("Minimax trên mọi từ: xấu nhất", max(stats), "| trung bình %.2f" % (sum(stats) / len(stats)))

Minimax trên mọi từ: xấu nhất 9 | trung bình 5.67


Code nộp LeetCode cho cách 5:

In [17]:
class SolutionMinimaxAll:
    def findSecretWord(self, words: List[str], master: 'Master') -> None:
        def match(a, b):
            return sum(x == y for x, y in zip(a, b))
        cand, guessed = list(words), set()
        while cand:
            if len(cand) == 1:
                g = cand[0]
            else:
                cset, best_key, g = set(cand), None, None
                for x in words:
                    cnt = [0] * 7
                    for w in cand:
                        if w != x:
                            cnt[match(x, w)] += 1
                    key = (max(cnt), x not in cset)
                    if best_key is None or key < best_key:
                        best_key, g = key, x
                if g in guessed:
                    g = cand[0]
            guessed.add(g)
            k = master.guess(g)
            if k == 6:
                return
            cand = [w for w in cand if w != g and match(w, g) == k]

for words, secret, allowed in EXAMPLES:
    m = Master(words, secret, allowed); SolutionMinimaxAll().findSecretWord(words, m); assert m.solved
print("SolutionMinimaxAll: OK")

SolutionMinimaxAll: OK


**Độ phức tạp cách 5:** mỗi lượt `O(n·c·L)`.

## 8. TRACE từng bước

Trace cách 3 (minimax) trên 100 từ ngẫu nhiên với một từ bí mật cố định: in số ứng viên, từ được chọn, điểm minimax, kết quả.

In [18]:
def trace_minimax(words, secret):
    m = Master(words, secret)
    cand = list(words)
    step = 0
    while cand:
        step += 1
        g = best_minimax(cand)
        cnt = [0] * 7
        for w in cand:
            if w != g:
                cnt[match(g, w)] += 1
        k = m.guess(g)
        print(f"lượt {step}: {len(cand):3} ứng viên | đoán '{g}' | phân nhóm {cnt} (xấu nhất {max(cnt)}) | nhận {k}")
        if k == 6:
            print("TRÚNG sau", step, "lượt;", m.verdict())
            return step
        cand = [w for w in cand if w != g and match(w, g) == k]
    return None

hard_secret = max(W100, key=lambda s: run(find_minimax, W100, s)[1])   # từ bí mật tốn nhiều lượt nhất
trace_minimax(W100, hard_secret)

lượt 1: 100 ứng viên | đoán 'jhccwb' | phân nhóm [66, 33, 0, 0, 0, 0, 0] (xấu nhất 66) | nhận 0
lượt 2:  66 ứng viên | đoán 'bvxxbr' | phân nhóm [45, 16, 4, 0, 0, 0, 0] (xấu nhất 45) | nhận 0
lượt 3:  45 ứng viên | đoán 'nipzus' | phân nhóm [31, 11, 2, 0, 0, 0, 0] (xấu nhất 31) | nhận 0
lượt 4:  31 ứng viên | đoán 'cqonni' | phân nhóm [20, 8, 2, 0, 0, 0, 0] (xấu nhất 20) | nhận 0
lượt 5:  20 ứng viên | đoán 'ouhfzj' | phân nhóm [13, 4, 2, 0, 0, 0, 0] (xấu nhất 13) | nhận 0
lượt 6:  13 ứng viên | đoán 'mawejf' | phân nhóm [7, 4, 1, 0, 0, 0, 0] (xấu nhất 7) | nhận 0
lượt 7:   7 ứng viên | đoán 'ayasow' | phân nhóm [3, 3, 0, 0, 0, 0, 0] (xấu nhất 3) | nhận 1
lượt 8:   3 ứng viên | đoán 'fxepoo' | phân nhóm [1, 0, 1, 0, 0, 0, 0] (xấu nhất 1) | nhận 0
lượt 9:   1 ứng viên | đoán 'akbrfg' | phân nhóm [0, 0, 0, 0, 0, 0, 0] (xấu nhất 0) | nhận 6
TRÚNG sau 9 lượt; You guessed the secret word correctly.


9

Bảng trạng thái (ví dụ 1 của đề, minimax):

| lượt | ứng viên | đoán | phân nhóm (0..6, trừ chính nó) | nhận |
|---|---|---|---|---|
| 1 | 4 | ccbazz (xấu nhất 2) | {2: 2, 3: 1} | 3 |
| 2 | 1 (acckzz) | acckzz | — | 6 ✔ |

In [19]:
trace_minimax(["acckzz", "ccbazz", "eiowzz", "abcczz"], "acckzz")

lượt 1:   4 ứng viên | đoán 'acckzz' | phân nhóm [0, 0, 1, 1, 1, 0, 0] (xấu nhất 1) | nhận 6
TRÚNG sau 1 lượt; You guessed the secret word correctly.


1

## 9. Code nộp LeetCode

Chọn minimax trên ứng viên (cách 3): tất định, đơn giản, chi phí nhỏ với `n <= 100`.

In [20]:
from typing import List

# class Master:   # do LeetCode cung cấp
#     def guess(self, word: str) -> int: ...

class Solution:
    def findSecretWord(self, words: List[str], master: 'Master') -> None:
        def match(a: str, b: str) -> int:
            return sum(x == y for x, y in zip(a, b))

        cand = list(words)
        while cand:
            # Chọn từ cực tiểu hoá kích thước nhóm lớn nhất (minimax)
            best, best_score = cand[0], len(cand) + 1
            for g in cand:
                cnt = [0] * 7
                for w in cand:
                    if w != g:
                        cnt[match(g, w)] += 1
                score = max(cnt)
                if score < best_score:
                    best, best_score = g, score
            k = master.guess(best)
            if k == 6:
                return
            # Chỉ giữ ứng viên nhất quán với kết quả
            cand = [w for w in cand if w != best and match(w, best) == k]

for words, secret, allowed in EXAMPLES:
    m = Master(words, secret, allowed)
    Solution().findSecretWord(words, m)
    assert m.solved, (words, secret)
    print(m.verdict(), "| số lần đoán:", m.calls)

You guessed the secret word correctly. | số lần đoán: 1
You guessed the secret word correctly. | số lần đoán: 1
You guessed the secret word correctly. | số lần đoán: 1


## 10. Vì sao thuật toán đúng

**Bất biến:** `secret ∈ cand` ở đầu mỗi vòng lặp.

- Ban đầu `cand = words ∋ secret`.
- Giả sử `secret ∈ cand` và ta đoán `g ∈ cand`, nhận `k = match(g, secret)`. Nếu `k = 6` thì `g = secret` (từ dài 6, khớp mọi vị trí) → thắng. Nếu `k < 6` thì `g ≠ secret` và `match(secret, g) = k`, nên `secret` được giữ lại trong `cand` mới.

**Kết thúc:** mỗi vòng loại ít nhất `g` khỏi `cand` (vì `w != g`), nên `|cand|` giảm ngặt; cùng với bất biến, thuật toán chắc chắn đoán trúng sau không quá `n` lượt.

**Về giới hạn lượt:** tính đúng ở trên không đảm bảo `<= allowedGuesses`. Minimax đảm bảo rằng sau mỗi lượt `|cand mới| <= min_g max_k |nhóm|`. Với từ ngẫu nhiên, giá trị này giảm nhanh, nên thực tế 100 từ cần trung bình khoảng 5–6 lượt, xấu nhất khoảng 9. Với dữ liệu đối kháng, không chiến lược nào đảm bảo tuyệt đối (ví dụ 100 từ chỉ khác nhau ở **một** vị trí → mọi lần đoán chỉ phân thành `{5: ...}` và trúng → cần tới 100 lượt) — đề đảm bảo test có lời giải hợp lý.

In [21]:
adversarial = ["aaaaa" + c for c in "abcdefghijklmnopqrstuvwxyz"]
print("Dữ liệu đối kháng (26 từ khác nhau đúng 1 vị trí):",
      "minimax xấu nhất =", max(run(find_minimax, adversarial, s, 100)[1] for s in adversarial), "lượt")

Dữ liệu đối kháng (26 từ khác nhau đúng 1 vị trí): minimax xấu nhất = 26 lượt


## 11. Phân tích Time/Space

Gọi `n` = số từ (≤ 100), `L = 6`, `G` = số lần đoán.

| Cách | Time | Space | Số lượt (100 từ ngẫu nhiên) |
|---|---|---|---|
| 1. Đoán lần lượt | `O(n·L)` | `O(1)` | tới 100 |
| 2. Ngẫu nhiên + lọc | `O(G·n·L)` | `O(n)` | TB ~6.5, có ván > 10 |
| 3. Minimax ứng viên | `O(G·n²·L)` | `O(n)` | TB ~5.7, xấu nhất ~9 |
| 4. Tần suất chữ cái | `O(G·n·L)` | `O(n)` | TB ~5.6, xấu nhất ~10 |
| 5. Minimax mọi từ | `O(G·n²·L)` | `O(n)` | TB ~5.7, xấu nhất ~9 |

In [22]:
import time

def evaluate(strategy, word_lists, allowed=10):
    worst, total, fails, games = 0, 0, 0, 0
    for words in word_lists:
        for s in words:
            ok, c = run(strategy, words, s, allowed)
            worst = max(worst, c); total += c; games += 1
            fails += (not ok)
    return worst, total / games, fails

rnd = random.Random(2024)
lists_full = [random_words(100, rnd) for _ in range(2)]
lists_small = [random_words(100, rnd, alpha="abcd") for _ in range(1)]     # chữ cái hẹp → khó hơn
for name, strat in [("random", find_random), ("minimax", find_minimax),
                    ("freq", find_freq), ("minimax_all", find_minimax_all)]:
    t0 = time.perf_counter()
    a = evaluate(strat, lists_full); b = evaluate(strat, lists_small)
    print(f"{name:12} 26 chữ: xấu nhất {a[0]}, TB {a[1]:.2f}, trượt {a[2]} | 4 chữ: xấu nhất {b[0]}, TB {b[1]:.2f}, trượt {b[2]}  ({(time.perf_counter() - t0):.1f}s)")

random       26 chữ: xấu nhất 13, TB 6.85, trượt 16 | 4 chữ: xấu nhất 6, TB 4.09, trượt 0  (0.1s)


minimax      26 chữ: xấu nhất 9, TB 5.71, trượt 0 | 4 chữ: xấu nhất 5, TB 3.86, trượt 0  (2.7s)
freq         26 chữ: xấu nhất 10, TB 5.58, trượt 0 | 4 chữ: xấu nhất 6, TB 3.93, trượt 0  (0.1s)


minimax_all  26 chữ: xấu nhất 9, TB 5.75, trượt 0 | 4 chữ: xấu nhất 5, TB 3.89, trượt 0  (3.6s)


## 12. Bộ test

In [23]:
strategies = {
    "random": find_random,
    "minimax": find_minimax,
    "freq": find_freq,
    "minimax_all": find_minimax_all,
}
classes = {
    "SolutionRandom": SolutionRandom,
    "SolutionMinimax": SolutionMinimax,
    "SolutionFreq": SolutionFreq,
    "SolutionMinimaxAll": SolutionMinimaxAll,
    "Solution": Solution,
}

# 12.1 Ví dụ đề + edge cases: mọi cách đều thắng
edge = EXAMPLES + [
    (["aaaaaa", "bbbbbb"], "bbbbbb", 10),
    (["abcdef", "abcdeg", "abcdeh"], "abcdeh", 10),
]
for words, secret, allowed in edge:
    for name, st in strategies.items():
        assert run(st, words, secret, allowed)[0], (name, words, secret)
    for name, cls in classes.items():
        m = Master(words, secret, allowed); cls().findSecretWord(words, m)
        assert m.solved, (name, words, secret)
print("Ví dụ + edge cases: OK")

# 12.2 Tính đúng (không giới hạn lượt): mọi chiến lược luôn tìm ra từ bí mật, không đoán từ ngoài danh sách
for seed in range(30):
    r_ = random.Random(seed)
    words = random_words(r_.randint(1, 40), r_, alpha="abc")
    for s in words:
        for name, cls in classes.items():
            m = Master(words, s, allowed_guesses=10 ** 9)
            cls().findSecretWord(words, m)
            assert m.solved and all(res != -1 for _, res in m.log), (name, words, s)
print("Luôn tìm ra từ bí mật (không giới hạn lượt): OK")

# 12.3 Giới hạn 10 lượt trên 100 từ ngẫu nhiên (26 chữ cái): Solution (minimax) thắng mọi ván
for words in lists_full:
    for s in words:
        m = Master(words, s, 10); Solution().findSecretWord(words, m)
        assert m.solved, s
print("Solution thắng mọi ván trong 10 lượt trên", len(lists_full) * 100, "ván ngẫu nhiên")

Ví dụ + edge cases: OK


Luôn tìm ra từ bí mật (không giới hạn lượt): OK


Solution thắng mọi ván trong 10 lượt trên 200 ván ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Lọc sai điều kiện (giữ từ có match **≥ k** thay vì **== k**)

**Code sai:**

In [24]:
def wrong_filter_ge(words, master):
    cand = list(words)
    while cand:
        g = best_minimax(cand)
        k = master.guess(g)
        if k == 6:
            return
        cand = [w for w in cand if w != g and match(w, g) >= k]    # SAI: >= thay vì ==

print("lọc >= : xấu nhất", max(run(wrong_filter_ge, W100, s)[1] for s in W100),
      "| lọc == :", max(run(find_minimax, W100, s)[1] for s in W100))

lọc >= : xấu nhất 23 | lọc == : 9


**Vì sao sai:** vẫn tìm ra bí mật (không loại nhầm), nhưng giữ lại quá nhiều ứng viên → tốn lượt. Từ bí mật có match **đúng bằng** `k`, không lớn hơn.

**Code đúng:** `match(w, g) == k`.

### Lỗi 2 — Không loại chính từ vừa đoán

**Code sai:**

In [25]:
def wrong_keep_guess(words, master, limit=50):
    cand = list(words)
    for _ in range(limit):                                        # chặn vòng lặp vô hạn để demo
        g = cand[0]
        k = master.guess(g)
        if k == 6:
            return
        cand = [w for w in cand if match(w, g) == k or w == g]   # SAI: giữ lại g
    return

m = Master(["aaaaaa", "aaaaab", "bbbbbb"], "bbbbbb", 10)
wrong_keep_guess(["aaaaaa", "aaaaab", "bbbbbb"], m)
print("số lần gọi:", m.calls, "| thắng?", m.solved, "| log:", m.log[:4])

số lần gọi: 50 | thắng? False | log: [('aaaaaa', 0), ('aaaaaa', 0), ('aaaaaa', 0), ('aaaaaa', 0)]


**Vì sao sai:** `g` vẫn nằm ở đầu danh sách nên lượt sau ta lại đoán `g`, mãi mãi (demo phải chặn bằng `limit`). **Code đúng:** luôn loại `g` (`w != g`). Lưu ý: với điều kiện `== k` và `k < 6`, `g` tự bị loại vì `match(g, g) = 6`; nhưng thêm `w != g` giúp code an toàn khi bạn đổi điều kiện lọc (ví dụ `>=` ở lỗi 1).

### Lỗi 3 — Đoán từ ngoài danh sách

**Code sai:**

In [26]:
def wrong_outside(words, master):
    master.guess("aaaaaa")                                        # SAI: lãng phí một lượt, nhận -1
    find_minimax(words, master)

worst_wrong = max(run(wrong_outside, W100, s, 10)[1] for s in W100)
worst_right = max(run(find_minimax, W100, s, 10)[1] for s in W100)
print("số lượt xấu nhất — có đoán từ ngoài danh sách:", worst_wrong, "| không:", worst_right)
print("nếu ngân sách chỉ còn", worst_right, "lượt: thắng", sum(run(wrong_outside, W100, s, worst_right)[0] for s in W100), "/ 100 vs",
      sum(run(find_minimax, W100, s, worst_right)[0] for s in W100), "/ 100")

số lượt xấu nhất — có đoán từ ngoài danh sách: 10 | không: 9


nếu ngân sách chỉ còn 9 lượt: thắng 95 / 100 vs 100 / 100


**Vì sao sai:** `-1` không cho thông tin gì nhưng vẫn tính lượt. **Code đúng:** chỉ đoán từ trong `words` (tốt nhất là trong `cand`).

### Lỗi 4 — Quên `return` sau khi trúng

Gọi thêm `guess` sau khi đã trúng không làm mất chiến thắng trong mô phỏng của ta, nhưng tốn công và có thể làm vượt lượt ở cài đặt chặt hơn. Luôn `return` ngay khi nhận `6`.

## 14. Mẹo & nhận dạng pattern

- **Bài tương tác "đoán với phản hồi"** → duy trì **tập ứng viên nhất quán** với mọi phản hồi đã nhận.
- Chọn câu hỏi bằng **minimax** (tối thiểu trường hợp xấu nhất) hoặc **kỳ vọng / entropy** (tối thiểu trung bình).
- Luôn tự dựng **lớp mô phỏng** để kiểm thử; đo **số lượt xấu nhất** trên mọi từ bí mật.
- Cố định **seed** cho các chiến lược ngẫu nhiên.
- Bài liên quan: **374 Guess Number Higher or Lower**, **375**, **Mastermind / Wordle** (minimax của Knuth), **1095 Find in Mountain Array** (tương tác), **278 First Bad Version**.

## 15. Biến thể

1. **Chọn theo entropy** (kỳ vọng thông tin): `g* = argmax Σ_k p_k·log(1/p_k)` với `p_k` = tỉ lệ nhóm `k`.
2. **Wordle**: phản hồi gồm cả "đúng chữ sai vị trí" → hàm phân nhóm khác, cùng khung minimax.
3. **Độ dài từ thay đổi / không cố định 6**: thay hằng số `6` bằng `len(secret)`.
4. **Tối thiểu số lượt trung bình** thay vì xấu nhất: chọn `g` cực tiểu `Σ |nhóm|²`.
5. **Tìm tối ưu tuyệt đối** (cây quyết định ít tầng nhất): tìm kiếm có cắt tỉa — chỉ khả thi với `n` nhỏ.

Minh hoạ biến thể 1 và 4:

In [27]:
import math

def best_entropy(cand):
    def H(g):
        cnt = Counter(match(g, w) for w in cand)
        n = len(cand)
        return -sum(c / n * math.log2(c / n) for c in cnt.values())
    return max(cand, key=lambda g: (H(g), g))

def best_sumsq(cand):
    return min(cand, key=lambda g: (sum(c * c for c in Counter(match(g, w) for w in cand if w != g).values()), g))

def make_finder(chooser):
    def finder(words, master):
        cand = list(words)
        while cand:
            g = chooser(cand)
            k = master.guess(g)
            if k == 6:
                return
            cand = [w for w in cand if w != g and match(w, g) == k]
    return finder

for name, ch in [("entropy", best_entropy), ("sum of squares", best_sumsq)]:
    w, avg, f = evaluate(make_finder(ch), lists_full[:1])
    print(f"{name:15}: xấu nhất {w}, trung bình {avg:.2f}, trượt {f}")

entropy        : xấu nhất 10, trung bình 5.51, trượt 0


sum of squares : xấu nhất 10, trung bình 5.66, trượt 0


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time/lượt | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đoán lần lượt | bỏ qua phản hồi | `O(L)` | `O(1)` | không bao giờ (chỉ minh hoạ) |
| 2. Ngẫu nhiên + lọc | lọc nhất quán | `O(nL)` | `O(n)` | dữ liệu ngẫu nhiên, code ngắn |
| 3. Minimax ứng viên | cực tiểu nhóm lớn nhất | `O(n²L)` | `O(n)` | **nộp bài** |
| 4. Tần suất chữ cái | điểm = Σ tần suất | `O(nL)` | `O(n)` | khi `n` lớn |
| 5. Minimax mọi từ | cho phép đoán từ đã loại | `O(n·c·L)` | `O(n)` | dữ liệu khó |

## 17. Bài tập

**Bài 1 (đoán output).** `match("abcdef", "abcxyz")`? `match("aaaaaa", "abcdef")`? `master.guess("zzzzzz")` khi `"zzzzzz"` không có trong `words`?

**Bài 2 (tính tay).** Với `cand = ["abcdef", "abcxyz", "uvwdef", "abxxef"]`, tính phân nhóm của từng từ và chọn từ minimax.

**Bài 3 (điền TODO).** Viết `worst_case_guesses(strategy, words)` trả về số lượt xấu nhất trên mọi từ bí mật.

**Bài 4 (tự viết lại).** Viết lại `Solution` dùng ma trận `M[i][j] = match(words[i], words[j])` tính trước một lần (`O(n²)`), các lượt sau chỉ tra bảng.

**Bài 5 (mini challenge).** Viết bộ tìm kiếm **tối ưu tuyệt đối** (số lượt xấu nhất nhỏ nhất) cho `n <= 8` bằng đệ quy + memo trên `frozenset` ứng viên, và so sánh với minimax tham lam.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def worst_case_guesses_todo(strategy, words):
    # TODO: max(run(strategy, words, s, 10**9)[1] for s in words)
    pass

# Bài 4 — TODO
def solution_matrix_todo(words, master):
    # TODO: M = [[match(a, b) for b in words] for a in words]; ứng viên là tập chỉ số
    pass

# Bài 5 — TODO
def optimal_worst_todo(words):
    # TODO: f(cand) = 1 + min_g max_k f(nhóm k) (nhóm 6 tốn 0 lượt thêm)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: đếm vị trí giống nhau.
- Mức 2: `abc` khớp, `def` vs `xyz` không.
- Mức 3: `3`, `1`, `-1`.

**Bài 2.**
- Mức 1: tính `match` cho 6 cặp.
- Mức 2: `abcdef`: với abcxyz 3, uvwdef 3, abxxef 4 → nhóm `{3: 2, 4: 1}`.
- Mức 3: `abcxyz` → `{3: 2, 0: 1}` (xấu nhất 2); `uvwdef` → `{3: 1, 0: 1, 2: 1}` (xấu nhất 1); `abxxef` → `{4: 1, 3: 1, 2: 1}` (xấu nhất 1). Từ đầu tiên đạt xấu nhất 1 theo thứ tự duyệt là `uvwdef`.

**Bài 3.**
- Mức 1: dùng hàm `run`.
- Mức 2: `allowed` rất lớn để không cắt.
- Mức 3: trả `max`.

**Bài 4.**
- Mức 1: làm việc với chỉ số thay cho chuỗi.
- Mức 2: `cnt[M[g][w]] += 1`.
- Mức 3: lọc `[w for w in cand if w != g and M[g][w] == k]`.

**Bài 5.**
- Mức 1: `f({x}) = 1`.
- Mức 2: với mỗi `g ∈ words`, nhóm theo `k`; nhóm `k = 6` chỉ có `g` (nếu `g` là ứng viên) → trúng, không tốn thêm.
- Mức 3: memo theo `frozenset`; so sánh với `worst_case_guesses(find_minimax, words)`.

## 19. Đáp án tham khảo

In [29]:
from functools import lru_cache

# Bài 1
assert match("abcdef", "abcxyz") == 3 and match("aaaaaa", "abcdef") == 1
assert Master(["abcdef"], "abcdef").guess("zzzzzz") == -1

# Bài 2
cand2 = ["abcdef", "abcxyz", "uvwdef", "abxxef"]
for g in cand2:
    print(g, dict(Counter(match(g, w) for w in cand2 if w != g)))
assert best_minimax(cand2) == "uvwdef"

# Bài 3
def worst_case_guesses(strategy, words):
    return max(run(strategy, words, s, 10 ** 9)[1] for s in words)

print("xấu nhất minimax trên W100:", worst_case_guesses(find_minimax, W100))

# Bài 4
def solution_matrix(words, master):
    n = len(words)
    M = [[match(a, b) for b in words] for a in words]
    cand = list(range(n))
    while cand:
        best, bs = cand[0], n + 1
        for g in cand:
            cnt = [0] * 7
            for w in cand:
                if w != g:
                    cnt[M[g][w]] += 1
            if max(cnt) < bs:
                best, bs = g, max(cnt)
        k = master.guess(words[best])
        if k == 6:
            return
        cand = [w for w in cand if w != best and M[best][w] == k]

for s in W100:
    assert run(solution_matrix, W100, s) == run(find_minimax, W100, s)

# Bài 5
def optimal_worst(words):
    words = tuple(words)
    @lru_cache(None)
    def f(cand):
        if len(cand) == 1:
            return 1
        best = float('inf')
        for g in words:
            groups = {}
            for w in cand:
                if w != g:
                    groups.setdefault(match(g, w), []).append(w)
            if not groups or (len(groups) == 1 and len(next(iter(groups.values()))) == len(cand)):
                continue                              # g không chia được gì → bỏ
            worst = max(f(frozenset(v)) for v in groups.values())
            best = min(best, 1 + worst)
        return best
    return f(frozenset(words))

for seed in range(15):
    r_ = random.Random(seed)
    ws = random_words(r_.randint(2, 7), r_, alpha="ab")
    opt = optimal_worst(ws)
    greedy = worst_case_guesses(find_minimax, ws)
    assert opt <= greedy
print("tối ưu vs minimax tham lam (ví dụ):", optimal_worst(random_words(7, random.Random(1), "ab")),
      worst_case_guesses(find_minimax, random_words(7, random.Random(1), "ab")))
print("Tất cả đáp án đúng")

abcdef {3: 2, 4: 1}
abcxyz {3: 2, 0: 1}
uvwdef {3: 1, 0: 1, 2: 1}
abxxef {4: 1, 3: 1, 2: 1}


xấu nhất minimax trên W100: 9


tối ưu vs minimax tham lam (ví dụ): 3 3
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (minimax + lọc nhất quán):**

```python
cand = list(words)
while cand:
    g = min(cand, key=lambda g: max(Counter(match(g, w) for w in cand if w != g).values(), default=0))
    k = master.guess(g)
    if k == 6: return
    cand = [w for w in cand if w != g and match(w, g) == k]
```

**Checklist trước khi submit:**

- [ ] Lọc bằng `match(w, g) == k` (đúng bằng, không phải `>=`).
- [ ] Loại chính từ vừa đoán.
- [ ] Chỉ đoán từ **trong danh sách** (tốt nhất trong ứng viên).
- [ ] `return` ngay khi nhận `6`.
- [ ] Kiểm thử bằng lớp `Master` mô phỏng trên **mọi** từ bí mật, đo số lượt xấu nhất.